# DQN: Deep Q-Network

**Paper**: Mnih et al., 2015 — *Human-level control through deep reinforcement learning* (DeepMind)

DQN uses a deep neural network to approximate the **action-value function** Q(s, a) — expected cumulative reward taking action *a* in state *s* then following the optimal policy.

**Bellman equation (TD target):**
$$Q(s,a) \leftarrow r + \gamma \max_{a'} Q(s', a')$$

Two key innovations for stable training:
1. **Experience Replay** — store $(s,a,r,s',\text{done})$ in a buffer, sample random mini-batches to break temporal correlations
2. **Target Network** — frozen copy $Q_{\text{target}}$ used for TD targets; synced from online every C steps to prevent chasing a moving target

```
Loss = E[ ( r + γ·max_a' Q_target(s',a') − Q_online(s,a) )² ]
```

Environment: **CartPole-v1** — balance a pole on a cart (state ∈ ℝ⁴, 2 discrete actions).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import gymnasium as gym
from collections import deque
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Q-Network

Maps state → Q-values for every action simultaneously. Output has one neuron per action — raw Q-values (no softmax).

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )

    def forward(self, x):
        return self.net(x)

## Replay Buffer

Stores transitions $(s, a, r, s', \text{done})$. Random sampling breaks correlation between consecutive steps, making gradient updates more like i.i.d. supervised learning.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=10_000):
        self.buf = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buf.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.LongTensor(actions).to(device),
            torch.FloatTensor(rewards).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(dones).to(device),
        )

    def __len__(self):
        return len(self.buf)

## DQN Agent

**ε-greedy exploration**: probability ε → random action, else greedy `argmax Q(s,·)`.  
ε decays exponentially: 1.0 → 0.01 over training so early exploration transitions to exploitation.

In [ ]:
class DQNAgent:
    def __init__(self, state_dim, action_dim,
                 lr=1e-3, gamma=0.99, batch_size=64,
                 buffer_size=10_000, target_update=100,
                 eps_start=1.0, eps_end=0.01, eps_decay=0.995):

        self.action_dim   = action_dim
        self.gamma        = gamma
        self.batch_size   = batch_size
        self.target_update = target_update
        self.eps          = eps_start
        self.eps_end      = eps_end
        self.eps_decay    = eps_decay
        self.step_count   = 0

        self.online_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = torch.optim.Adam(self.online_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)

    def select_action(self, state):
        if random.random() < self.eps:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(device)
            return self.online_net(s).argmax(1).item()

    def update(self):
        if len(self.buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)

        # Q(s, a) from online network
        q_values = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # TD target: r + γ * max_a' Q_target(s', a')
        with torch.no_grad():
            next_q  = self.target_net(next_states).max(1)[0]
            target  = rewards + self.gamma * next_q * (1 - dones)

        loss = F.mse_loss(q_values, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online_net.parameters(), 10.0)
        self.optimizer.step()

        # Decay ε
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

        # Sync target network periodically
        self.step_count += 1
        if self.step_count % self.target_update == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())

        return loss.item()

## Training

In [ ]:
env = gym.make('CartPole-v1')
state_dim  = env.observation_space.shape[0]   # 4
action_dim = env.action_space.n               # 2

agent = DQNAgent(state_dim, action_dim)

EPISODES = 400
episode_rewards = []

for ep in range(EPISODES):
    state, _ = env.reset(seed=ep)
    total_reward = 0
    while True:
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.buffer.push(state, action, reward, next_state, float(done))
        agent.update()
        state = next_state
        total_reward += reward
        if done:
            break
    episode_rewards.append(total_reward)
    if (ep + 1) % 50 == 0:
        avg = np.mean(episode_rewards[-50:])
        print(f'Ep {ep+1:4d}/{EPISODES}  Avg(50): {avg:6.1f}  ε: {agent.eps:.3f}')

env.close()

## Results

In [ ]:
def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, alpha=0.3, color='steelblue', label='Reward')
plt.plot(np.arange(len(smooth(episode_rewards)))+19, smooth(episode_rewards),
         color='steelblue', linewidth=2, label='Smoothed (20-ep)')
plt.axhline(195, color='red', linestyle='--', label='Solved (195)')
plt.xlabel('Episode'); plt.ylabel('Total Reward')
plt.title('DQN on CartPole-v1')
plt.legend(); plt.tight_layout()
plt.show()
print(f'Best avg (last 50): {np.mean(episode_rewards[-50:]):.1f}')

## Evaluate

In [ ]:
agent.eps = 0.0
agent.online_net.eval()
eval_env = gym.make('CartPole-v1')
eval_rewards = []
for _ in range(20):
    s, _ = eval_env.reset()
    total = 0
    while True:
        a = agent.select_action(s)
        s, r, term, trunc, _ = eval_env.step(a)
        total += r
        if term or trunc:
            break
    eval_rewards.append(total)
eval_env.close()
print(f'Greedy eval (20 ep): mean={np.mean(eval_rewards):.1f}  std={np.std(eval_rewards):.1f}')

## Summary

| Component | Detail |
|-----------|--------|
| **Q-Network** | MLP: 4→128→128→2 |
| **Replay Buffer** | 10,000 transitions, random mini-batch |
| **Target Network** | Hard sync every 100 gradient steps |
| **Exploration** | ε-greedy: 1.0→0.01, decay ×0.995/step |
| **Loss** | MSE Bellman error |
| **Optimizer** | Adam lr=1e-3 |
| **Known issue** | Q-value overestimation (max over noisy Q) → fixed by DDQN |